In [1]:
import torch
from itertools import chain
from transformers import pipeline
import json
from pathlib import Path
import numpy as np



In [ ]:
class YearWalk:
    """
    A class to represent all geographical references from a given title a given year.

    Attributes
    ----------
    path : str
        The path of the file to be processed.
    
    Methods
    -------
    info(additional=""):
        Stores the references with associated title and the number of error words.
    """
    def __init__(self, path):
        """
        Constructs the necessary parameters and stores the relevant information.

        Parameters
        ----------
        path : str
          The path of the file to be processed.
        title : str
          Represents the title of the newspaper. Extracted from the path if filename based on the convention 'Word_sum [title]_partX.txt'.
        loclist : list[str]
          Lists all the geographical references identified with NER.
        errors : int
          Counts the number of word that are too long to lead to RuntimeError.
        """
        self.path = path
        self.title = ""
        self.loclist = []
        self.errors = 0

        # Load model on GPU if available. 
        device = 0 if torch.cuda.is_available() else -1
        self.nlp = pipeline("ner", model="dslim/bert-base-NER", device=device)

        # Read and split text into a list of separate words.
        with open(self.path, "r", encoding="utf8") as f:
            data = f.read()
        words = data.split(" ")

        # Batch processing
        for i in range(0, len(words), batch_size):
            batch = words[i:i+batch_size]
            
            #Looks through the words for named entities.
            try:
                results = self.nlp(batch)

                # Flatten the list of lists
                flat_results = list(chain.from_iterable(results))
                
                #Picks out b-loc results ("beginning-loc", first word in location name) and add to the list of locations.
                for term in flat_results:
                  if term["entity"].startswith("B-LOC"):
                    word = term["word"]
                    self.loclist.append(word)
                  
                  #In case there is additional words part of the same name ("inside-loc"), those are added to the former entry
                  elif term["entity"].startswith("I-LOC"):
                    word = term["word"]
                      try:
                        self.loclist[-1] += " " + word
                      
                      #It happens that the first loc is identified as an i-loc, and thus being unable t identify self.loclist[-1]
                      except IndexError:
                        print("Problem occurred in " + self.path[-16:])
            
            #The way I am splitting words leave a lot of long nonsense words, whose origin I haven't been able to identify. It varies a lot from
            #title to title, spanning from about twenty to many hundreds. Still, it is such a small number of the total ammount of words that
            #I have not felt motivated in adressing it for this project but that may be 1) too lazy of me and 2) unnecessary as there might be
            #an easy solution. For the time being, I print the word and count the number of occurances to just keep tabs on the issue in case
            #it sould get out of hand or clear patterns in the type of words emerge.
            except RuntimeError:
                print("Batch starting with " + batch[0][:50] + "... had a too long tensor.")
                self.errors += 1
        pass
  
class LocYear:
  """
  Class that represents all identified locations across all titles in a given year.

  Parameters
  ----------
  path : str
     The path should lead to a directory named {year} inside a directory called "Words". The year directory contains files named according to the convention 'Word_sum {title}' of
     of the nine titles and produced in accordance with the WordSum.survey() function.

  Methods
    -------
    locations():
        Stores the locations identified over all titles in a .txt file.
  """
  def __init__(self, path):
    """
    Constructs the necessary parameters.

    Parameters
    ----------
    path : str
        Path to the directory of files to be processed.
    year : str
        Identifies the year from the path.
    locs : list
        Master list of all the locations from a given year.    
    """
    self.path = path
    self.year = ""
    self.locs = []

    #Splits the path to extract the year.
    pathparts = self.path.split("Words")
    pathend = pathparts[1]
    self.year = pathend[1:]
    pass

  def locations(self):
    """
    Adds the locations and stores them in a .txt file.

    Returns
    -------
    None
    """
    
    #Uses Path to read and iterate over the directory, treating each file (=each title) as a YearWalk object. The locations identified
    #in each file is then extended to the master list of the LocYear. 
    directory = Path(self.path)
    for file in directory.iterdir():
      if file.is_file():
        str_path = str(file)
        filerecord = YearWalk(str_path)
        self.locs.extend(filerecord.loclist)

        #In order to save the process regularly, I have set the process to overwrite the .txt file after every title.
        with open(fr'~\{self.year}.txt', 'w') as convert_file:
          convert_file.write(json.dumps(self.locs))
        
        #When a title/file is processed, this little printout marks the progress and notes the number of error words so I can store
        #keep records.
        print(f"""
            
          ///    Number of errorwords: {filerecord.errors}    \\\
          \\\    {filerecord.title} is done!                  ///
          
          """)
    pass
   

<>:62: SyntaxWarning: invalid escape sequence '\ '
<>:62: SyntaxWarning: invalid escape sequence '\ '
C:\Users\jensn\AppData\Local\Temp\ipykernel_16720\2278551993.py:62: SyntaxWarning: invalid escape sequence '\ '
  """)


In [ ]:
test = LocYear(r"~\Words\1866")
test.locations()


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


siiderhuskkMvvslrtiugf\u00f6h\u00f6gagfvksmklskhtr... had a too long tensor.
otdmBthnlehapnstadstdedef\u00f6r\u00e4laHemicchbed... had a too long tensor.
\"Bsor\u00e4mvifj\u00e4anh\u00f6flef\u00e5qvfuterp... had a too long tensor.
etenfraskomomdeh\u00e4setaltilF\u00f6ocm\u00f6fsig... had a too long tensor.
tikpodevaf\u00f6rstosatinreostilgemeupatfraatvimgi... had a too long tensor.
offb\u00e4sh\u00e5skre\u00f6f\u00e5tBeStNr\u00e5be... had a too long tensor.
fbvs\u00e4lariodudocremr\u00e4edl\u00e4gohvdbydeDr... had a too long tensor.
g\u00f6PrimeZeinvF\u00f6delParandanlarfarnantyging... had a too long tensor.
o\u00e4d\u00e4snnsodnfsiadDmvsdsnvnftsperl\u00e5d\... had a too long tensor.
tsSKjG4lksOWISvOsPdpSstsjpmtiBrefroamn\u00e5ttadeo... had a too long tensor.
HochGroseomaLonMLunMHanJ\u00d6Jt\u00e4IngeBrutare\... had a too long tensor.
sainRy\u00f6fvagesainRy\u00f6fvagesaRy\u00f6fvages... had a too long tensor.
hokaleskv\u00f6etDcenevekaleskv\u00f6etDcenevelesk... had a too long tensor.

TypeError: can only concatenate str (not "int") to str